# 05 — External Data Features (MODIS LST only)

Integrates one external dataset:

| Source | Variables → Features |
|---|---|
| **MODIS LST** (8-day, 1 km) | LST_emergence, LST_winter_mean, LST_jointing, LST_anthesis_max, LST_night_winter_min |

> **2026 update — slim 3-source stack.** A per-source ablation
> (`data/results/per_source_ablation_summary.csv`) showed that **SoilGrids**
> static soil properties and **SMAP** soil-moisture features added
> negligible R² gain (mean ΔR² ≈ 0 across 8 stages) over the
> HLS + Daymet + MODIS LST core. Only **MODIS LST** contributed meaningful
> signal (mean ΔR² = +0.019, including +0.083 on tillering, +0.029 on
> maturity, +0.023 on anthesis). SoilGrids and SMAP were therefore removed
> from the active pipeline.

**Why MODIS LST is kept:** it captures *surface* temperature (canopy /
bare-soil radiative T) that is structurally different from the *air* T
provided by Daymet. The `LST − T_air` gap is a direct drought-stress signal
(stomata-closed canopies stop transpiring → leaf temperature jumps 10–20 °C
above ambient), and bare-soil heating in early-season is what drives much
of the tillering R² gain.


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Slim 3-source stack: only MODIS LST is consumed here. SoilGrids and SMAP
# were ablation-removed (mean ΔR² ≈ 0); see archive folder for the dropped
# extraction scripts and the explanation in this notebook's first cell.
PATHS = {
    'feat_in':  'data/processed/features/features_stage_specific.parquet',
    'feat_out': 'data/processed/features/features_with_external.parquet',
    'modis_lst':'data/raw/satellite/modis_lst_buffer.csv',
}

feat = pd.read_parquet(PATHS['feat_in'])
feat['field_id'] = feat['field_id'].astype(str)
feat['year'] = feat['year'].astype(int)
print(f'Stage-specific features (input): {feat.shape}')


## 1. MODIS LST — daily surface temperature aggregated per stage window


In [ ]:
lst = pd.read_csv(PATHS['modis_lst'])
lst['date'] = pd.to_datetime(lst['date'])
lst['field_id'] = lst['FIELDID'].astype(str)

lst['cy'] = lst['date'].dt.year
lst['month'] = lst['date'].dt.month
lst['gs_start_year'] = np.where(lst['month'] >= 7, lst['cy'], lst['cy'] - 1)
lst['harvest_year']  = lst['gs_start_year'] + 1
lst['gs_start_date'] = pd.to_datetime(lst['gs_start_year'].astype(str) + '-07-01')
lst['dos'] = (lst['date'] - lst['gs_start_date']).dt.days + 1

print(f'MODIS LST: {lst.shape}, unique fields: {lst["field_id"].nunique()}')

LST_WINDOWS = {
    'lst_day_emergence':    {'col': 'lst_day_C',   'agg': 'mean', 'dos': (90, 150)},
    'lst_day_winter_mean':  {'col': 'lst_day_C',   'agg': 'mean', 'dos': (150, 220)},
    'lst_day_jointing':     {'col': 'lst_day_C',   'agg': 'mean', 'dos': (240, 290)},
    'lst_day_anthesis_max': {'col': 'lst_day_C',   'agg': 'max',  'dos': (290, 330)},
    'lst_night_winter_min': {'col': 'lst_night_C', 'agg': 'min',  'dos': (150, 220)},
}

lst_records = []
fy_pairs = feat[['field_id','year']].drop_duplicates().values
for fid, yr in fy_pairs:
    fid = str(fid); yr = int(yr)
    sub = lst[(lst['field_id']==fid) & (lst['harvest_year']==yr)]
    row = {'field_id': fid, 'year': yr}
    for fname, spec in LST_WINDOWS.items():
        win = sub[(sub['dos']>=spec['dos'][0]) & (sub['dos']<=spec['dos'][1])][spec['col']].dropna()
        if len(win) == 0:
            row[fname] = np.nan
        elif spec['agg'] == 'mean':
            row[fname] = float(win.mean())
        elif spec['agg'] == 'max':
            row[fname] = float(win.max())
        elif spec['agg'] == 'min':
            row[fname] = float(win.min())
    lst_records.append(row)

lst_features = pd.DataFrame(lst_records)
print(f'\nLST features computed: {lst_features.shape}')
print(lst_features.describe().round(2))


## 2. Merge LST features and save


In [ ]:
lst_features['field_id'] = lst_features['field_id'].astype(str)
lst_features['year'] = lst_features['year'].astype(int)
feat_v5 = feat.merge(lst_features, on=['field_id','year'], how='left')

new_cols = list(LST_WINDOWS.keys())
print('Correlations of new MODIS LST features with flag_true_doy:')
for c in new_cols:
    sub = feat_v5[[c, 'flag_true_doy']].dropna()
    if len(sub) < 50:
        print(f'  {c:30s}  n={len(sub):4d}  insufficient')
        continue
    r = np.corrcoef(sub[c], sub['flag_true_doy'])[0,1]
    print(f'  {c:30s}  n={len(sub):4d}  r={r:+.3f}')

feat_v5.to_parquet(PATHS['feat_out'], index=False)
print(f'\nSaved: {PATHS["feat_out"]}')
print(f'Shape: {feat_v5.shape}')
print(f'New columns added: {len(new_cols)}')
